# MaPLe: Multi-modal Prompt Learning trên Fruit Dataset (Kaggle)

Notebook này thiết lập quy trình hoàn chỉnh để huấn luyện và đánh giá **MaPLe (CVPR 2023)** trên bộ dữ liệu Fruit, đồng thời so sánh trực diện với các mô hình trước đó (**SWAT**, **AGNN**, và **Zero-shot CLIP**).

### Quy trình thực hiện:
1. **Cài đặt môi trường**: PyTorch, Dassl.pytorch, và các phụ thuộc.
2. **Cấu hình đường dẫn dataset**: Tự động nhận diện dataset trong `/kaggle/input/`.
3. **Baseline Zero-Shot CLIP**: Đánh giá CLIP ViT-B/16 nguyên bản.
4. **Huấn luyện MaPLe**: Học multimodal prompt trên 14 lớp Base (5-shot hoặc 16-shot, 3 seeds).
5. **Đánh giá Base-to-Novel**: Base Accuracy, Novel Accuracy, Harmonic Mean (HM).
6. **Đánh giá Episodic 5-way 5-shot**: 600 episodes, ProtoNet cosine metric (so sánh trực tiếp với kết quả SWAT `78.00 ± 0.44%` và AGNN).
7. **Bảng tổng hợp so sánh** giữa các phương pháp.

> **Lưu ý cài đặt Kaggle**:
> - **Accelerator**: GPU T4 x2 hoặc P100 (Settings -> Accelerator -> GPU)
> - **Internet**: ON (Settings -> Internet -> On)

## 0. Kiểm tra GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found!')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device name : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Cài đặt thư viện & Dassl.pytorch

In [ ]:
# 1. Cài đặt các gói phụ trợ
!pip install -q ftfy regex yacs gdown

# 2. Cài đặt Dassl.pytorch từ GitHub
import os, sys
DASSL_DIR = '/kaggle/working/Dassl.pytorch'
if not os.path.exists(DASSL_DIR):
    !git clone https://github.com/KaiyangZhou/Dassl.pytorch.git {DASSL_DIR}
    !cd {DASSL_DIR} && pip install -q -r requirements.txt && python setup.py develop
    print('Dassl.pytorch installed successfully.')
else:
    print('Dassl.pytorch directory already exists.')

# Đảm bảo Python nhận diện Dassl ngay lập tức kể cả khi không restart session
if DASSL_DIR not in sys.path:
    sys.path.insert(0, DASSL_DIR)

# Kiểm tra import Dassl
try:
    import dassl
    print(f'Dassl loaded from: {dassl.__file__}')
except ImportError as e:
    print(f'Warning: {e}. Added {DASSL_DIR} to sys.path.')

## 2. Thiết lập thư mục làm việc MaPLe

In [ ]:
import os, sys

REPO_URL = 'https://github.com/nta2112/MaPle-for-Fruit-classification.git'
MAPLE_DIR = '/kaggle/working/MaPle-for-Fruit-classification'

if not os.path.exists(MAPLE_DIR):
    !git clone {REPO_URL} {MAPLE_DIR}
else:
    print('Repo already exists, pulling latest updates...')
    !git -C {MAPLE_DIR} pull

os.chdir(MAPLE_DIR)
sys.path.insert(0, MAPLE_DIR)
print(f'Current working directory: {os.getcwd()}')
!ls -la

## 3. Cấu hình đường dẫn Dataset trên Kaggle

> Tự động dò tìm dataset trong `/kaggle/input/`. Nếu tên dataset khác, bạn chỉ cần sửa `DATASET_SLUG`.

In [ ]:
import os, glob, json

print('Datasets in /kaggle/input:')
for d in sorted(os.listdir('/kaggle/input')):
    print(f'  /kaggle/input/{d}')

# Tên slug của dataset trên Kaggle của bạn (thay đổi nếu cần)
DATASET_SLUG = 'fruit-recognition'

# Tự động tìm đường dẫn ảnh và split
possible_dataset_paths = [
    f'/kaggle/input/{DATASET_SLUG}/archive/images/images',
    f'/kaggle/input/{DATASET_SLUG}/images/images',
    f'/kaggle/input/{DATASET_SLUG}/images',
]
DATASET_PATH = None
for p in possible_dataset_paths:
    if os.path.exists(p):
        DATASET_PATH = p
        break

if DATASET_PATH is None:
    # Tìm bất kỳ thư mục con nào chứa class 'america_apple'
    found = glob.glob('/kaggle/input/**/america_apple', recursive=True)
    if found:
        DATASET_PATH = os.path.dirname(found[0])

possible_splits = [
    f'/kaggle/input/{DATASET_SLUG}/test_split.json',
    f'/kaggle/input/{DATASET_SLUG}/split.json',
]
SPLIT_PATH = None
for s in possible_splits:
    if os.path.exists(s):
        SPLIT_PATH = s
        break

if SPLIT_PATH is None:
    found_split = glob.glob('/kaggle/input/**/test_split.json', recursive=True)
    if found_split:
        SPLIT_PATH = found_split[0]

print(f'\nSelected DATASET_PATH : {DATASET_PATH}')
print(f'Selected SPLIT_PATH   : {SPLIT_PATH}')

assert DATASET_PATH and os.path.exists(DATASET_PATH), f'Cannot find images folder!'
assert SPLIT_PATH and os.path.exists(SPLIT_PATH), f'Cannot find test_split.json!'

with open(SPLIT_PATH) as f:
    split_data = json.load(f)
print(f'\nTrain classes ({len(split_data["train"])}): {split_data["train"]}')
print(f'Val classes   ({len(split_data["val"])}): {split_data["val"]}')
print(f'Test classes  ({len(split_data["test"])}): {split_data["test"]}')

## 4. Baseline: Zero-Shot CLIP (ViT-B/16)

Đánh giá độ chính xác của mô hình CLIP ViT-B/16 gốc (chưa qua fine-tune/prompt learning) theo giao thức episodic 5-way 5-shot trên 5 lớp test.

In [ ]:
!python eval_maple_episodic.py \
    --zeroshot \
    --split-path "{SPLIT_PATH}" \
    --dataset-path "{DATASET_PATH}" \
    --split test \
    --n_episodes 600 \
    --n_way 5 --k_shot 5 --n_query 15 \
    --seed 42 \
    --feat-cache /kaggle/working/cache_zeroshot_vitb16.pth \
    --output-json /kaggle/working/result_zeroshot.json

## 5. Huấn luyện MaPLe trên 14 lớp Base (3 Seeds)

- **Backbone**: ViT-B/16
- **Prompt Depth**: 9
- **Context Tokens**: 2 (`a photo of a`)
- **Shots**: 16 (hoặc 5)
- **Epochs**: 5 (~2-3 phút mỗi seed trên GPU T4)

In [ ]:
SHOTS = 5  # Hoặc đặt = 5 nếu muốn đúng số shot với SWAT
SEEDS = [1, 2, 3]

for seed in SEEDS:
    print(f'\n' + '=' * 60)
    print(f'Training MaPLe - Seed {seed} ({SHOTS}-shot)')
    print('=' * 60)
    
    output_dir = f'/kaggle/working/output/base2new/train_base/fruit/shots_{SHOTS}/MaPLe/vit_b16_c2_ep5_batch4_2ctx/seed{seed}'
    
    !python train.py \
        --root "{DATASET_PATH}" \
        --split-path "{SPLIT_PATH}" \
        --seed {seed} \
        --trainer MaPLe \
        --dataset-config-file configs/datasets/fruit.yaml \
        --config-file configs/trainers/MaPLe/vit_b16_c2_ep5_batch4_2ctx.yaml \
        --output-dir "{output_dir}" \
        DATASET.NUM_SHOTS {SHOTS} \
        DATASET.SUBSAMPLE_CLASSES base

print('All seeds trained successfully!')

## 6. Đánh giá Base-to-Novel Generalization (CVPR 2023 Protocol)

Đánh giá trên tập test của lớp **Base** và tập test của lớp **Novel** để tính **Harmonic Mean**.

In [ ]:
import re
base2new_results = {}

for seed in SEEDS:
    model_dir = f'/kaggle/working/output/base2new/train_base/fruit/shots_{SHOTS}/MaPLe/vit_b16_c2_ep5_batch4_2ctx/seed{seed}'
    dir_base  = f'/kaggle/working/output/base2new/test_base/fruit/shots_{SHOTS}/MaPLe/vit_b16_c2_ep5_batch4_2ctx/seed{seed}'
    dir_new   = f'/kaggle/working/output/base2new/test_new/fruit/shots_{SHOTS}/MaPLe/vit_b16_c2_ep5_batch4_2ctx/seed{seed}'
    
    print(f'\n--- Evaluating Seed {seed} on BASE classes ---')
    !python train.py \
        --root "{DATASET_PATH}" \
        --split-path "{SPLIT_PATH}" \
        --seed {seed} \
        --trainer MaPLe \
        --dataset-config-file configs/datasets/fruit.yaml \
        --config-file configs/trainers/MaPLe/vit_b16_c2_ep5_batch4_2ctx.yaml \
        --output-dir "{dir_base}" \
        --model-dir "{model_dir}" \
        --load-epoch 5 \
        --eval-only \
        DATASET.NUM_SHOTS {SHOTS} \
        DATASET.SUBSAMPLE_CLASSES base

    print(f'\n--- Evaluating Seed {seed} on NOVEL classes ---')
    !python train.py \
        --root "{DATASET_PATH}" \
        --split-path "{SPLIT_PATH}" \
        --seed {seed} \
        --trainer MaPLe \
        --dataset-config-file configs/datasets/fruit.yaml \
        --config-file configs/trainers/MaPLe/vit_b16_c2_ep5_batch4_2ctx.yaml \
        --output-dir "{dir_new}" \
        --model-dir "{model_dir}" \
        --load-epoch 5 \
        --eval-only \
        DATASET.NUM_SHOTS {SHOTS} \
        DATASET.SUBSAMPLE_CLASSES new

## 7. Đánh giá Episodic 5-way 5-shot (So sánh trực tiếp với SWAT & AGNN)

Chạy 600 episodes trên 5 lớp test: `korea_orange`, `chinese_apple`, `chinese_grape`, `southafrica_apple`, `netherlands_apple`.

In [ ]:
episodic_results = {}

for seed in SEEDS:
    model_dir = f'/kaggle/working/output/base2new/train_base/fruit/shots_{SHOTS}/MaPLe/vit_b16_c2_ep5_batch4_2ctx/seed{seed}'
    feat_cache = f'/kaggle/working/cache_maple_seed{seed}.pth'
    out_json = f'/kaggle/working/result_maple_seed{seed}.json'
    
    print(f'\n' + '=' * 60)
    print(f'Episodic 5-way 5-shot Evaluation - Seed {seed}')
    print('=' * 60)
    
    !python eval_maple_episodic.py \
        --model-dir "{model_dir}" \
        --load-epoch 5 \
        --split-path "{SPLIT_PATH}" \
        --dataset-path "{DATASET_PATH}" \
        --split test \
        --n_episodes 300 \
        --n_way 5 --k_shot 5 --n_query 15 \
        --seed 42 \
        --feat-cache "{feat_cache}" \
        --output-json "{out_json}"
    
    if os.path.exists(out_json):
        with open(out_json) as f:
            data = json.load(f)
        episodic_results[seed] = (data['mean_acc'], data['ci95'])

## 8. Bảng Tổng Hợp So Sánh Kết Quả

Tổng hợp kết quả giữa các mô hình trên bộ dữ liệu Fruit.

In [ ]:
import numpy as np, json

print('=' * 65)
print('  FINAL EPISODIC 5-WAY 5-SHOT COMPARISON ON FRUIT DATASET')
print('=' * 65)

# Đọc Zero-shot
zs_mean, zs_ci = None, None
if os.path.exists('/kaggle/working/result_zeroshot.json'):
    with open('/kaggle/working/result_zeroshot.json') as f:
        zs_data = json.load(f)
    zs_mean, zs_ci = zs_data['mean_acc'], zs_data['ci95']

if episodic_results:
    accs = [v[0] for v in episodic_results.values()]
    maple_mean = np.mean(accs)
    maple_ci = np.mean([v[1] for v in episodic_results.values()])
    print(f'MaPLe Seeds:')
    for s, (m, c) in sorted(episodic_results.items()):
        print(f'  Seed {s}: {m:.2f} +/- {c:.2f}%')
    print(f'MaPLe Average: {maple_mean:.2f} +/- {maple_ci:.2f}%')

print('\n' + '-' * 65)
print(f'| {"Model":<25} | {"Method":<20} | {"5-way 5-shot Acc":<15} |')
print('-' * 65)
if zs_mean is not None:
    print(f'| {"CLIP ViT-B/16":<25} | {"Zero-shot":<20} | {zs_mean:>6.2f} +/- {zs_ci:.2f}%     |')
print(f'| {"SWAT (ViT-B/32)":<25} | {"Fine-tune (5-shot)":<20} | {"78.00 +/- 0.44%":<15} |')
print(f'| {"AGNN (ResNet12)":<25} | {"Episodic GNN":<20} | {"--.-- +/- -.--%":<15} |')
if episodic_results:
    print(f'| {"MaPLe (ViT-B/16)":<25} | {"Multimodal Prompt":<20} | {maple_mean:>6.2f} +/- {maple_ci:.2f}%     |')
print('=' * 65)